<a href="https://colab.research.google.com/github/annagiacometti/cultural-data-framework-photography-festivals/blob/main/estrazione_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch requests beautifulsoup4 pandas

L'agente sta usando: cpu
Caricamento del modello Flan-T5 in corso...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Agente pronto. La pipeline manuale è configurata.


# 1. Prompt zero-shot

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. CONFIGURAZIONE DELL'AMBIENTE (PERCEZIONE COMPUTAZIONALE)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/flan-t5-large" # Modello Open-Source per garantire la sovranità del dato

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

# 2. FASE DI HARVESTING: ROTTURA DEI SILOS INFORMATIVI
# L'agente raccoglie frammenti da diverse fonti per ricostruire l'unità dell'oggetto culturale
urls = [
    "https://www.festivaldellafotografiaetica.it/",
    "https://www.festivaldellafotografiaetica.it/organizzazione/",
    "https://www.festivaldellafotografiaetica.it/world-report-award-it/"
]

def scrape_silos(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, 'html.parser')
    # Pulizia del rumore: eliminiamo header, footer e script per isolare il 'segnale'
    for s in soup(['script', 'style', 'nav', 'footer', 'header']): s.extract()
    return soup.get_text(separator=' ', strip=True)

# Unificazione dei testi in un unico 'Corpus Sensibile'
corpus_totale = " ".join([scrape_silos(u) for u in urls])

# 3. FASE DI NORMALIZZAZIONE: APPLICAZIONE DEL FRAMEWORK EUROPEO
# Definiamo gli attributi target secondo la tassonomia della tesi
framework_attributi = {
    "festival_name": "Full name of the festival",
    "city": "City where it takes place",
    "region": "Italian region",
    "start_year": "Year of the first edition",
    "organizer_type": "Legal nature (non-profit, public, or private)",
    "international": "International orientation? (Yes/No)",
    "public_funding": "Presence of public funding? (Yes/No)",
    "ticket_policy": "Entry type (Free/Paid/Mixed)"
}

riga_dataset = {}

# 4. INFERENZA: PROCEDURA DI SODDISFAZIONE (SATISFICING)
# L'agente applica l'Attenzione Selettiva su ogni variabile del framework
print("Estrazione in corso...")
for chiave, domanda in framework_attributi.items():
    # Limitiamo il contesto per rispettare la memoria di lavoro del modello (512 token)
    prompt = f"Context: {corpus_totale[:2000]} \n Question: {domanda} \n Answer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20)

    # Decodifica del risultato e inserimento nel dataset
    risposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    riga_dataset[chiave] = risposta

# 5. OUTPUT FINALE: DALLA DERIVA ALL'ORDINE (PANDAS DATAFRAME)
df_final = pd.DataFrame([riga_dataset])
print("\n--- RISULTATO DELLA MAPPATURA SCIENTIFICA ---")
print(df_final.to_string())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (725 > 512). Running this sequence through the model will result in indexing errors


Estrazione in corso...

--- RISULTATO DELLA MAPPATURA SCIENTIFICA ---
                     festival_name  city region start_year organizer_type international public_funding ticket_policy
0  Festival della Fotografia Etica  Lodi   Lodi       2010     Non-profit            No             No          Paid


Il processo inizia con quella che tecnicamente chiamiamo fase di Harvesting (raccolta). L'obiettivo è "rompere i silos": le informazioni su un festival non si trovano mai in un unico punto, ma sono frammentate tra vari indirizzi URL. Attraverso lo scraping, abbiamo rimosso il "rumore" del sito (menu, pubblicità, codici tecnici) per isolare il segnale pulito, ovvero il testo nudo e crudo.

Una volta ottenuto questo corpus di testo, abbiamo interrogato il modello Flan-T5-large. In questa fase abbiamo utilizzato un approccio Zero-Shot: abbiamo cioè chiesto alla macchina di estrarre dati complessi (come la natura legale dell'organizzatore o la politica dei biglietti) senza fornirle alcun esempio precedente, basandoci esclusivamente sulla sua capacità di comprensione linguistica immediata.

Strumenti:
- BeautifulSoup (Il Filtro): È lo strumento di analisi dell'HTML
- Flan-T5-large (Il Cervello Linguistico): È un modello Large Language Model (LLM) di tipo Encoder-Decoder.

Criticità:
- errori di classificazione es. Lodi al posto di Lombardia (SOLUZIONE: approccio few-shot)
- mancanza di informazioni negli URL dati in input (SOLUZIONE: inserire più URL)
- limite della memoria di lavoro del modello (limitata a 512 token (circa 400 parole))

# 2. Prompt few-shot

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. CONFIGURAZIONE MODELLO (PERCEZIONE COMPUTAZIONALE)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

# 2. FASE DI HARVESTING: AMPLIAMENTO DEI SILOS INFORMATIVI
urls = [
    "https://www.festivaldellafotografiaetica.it/",
    "https://www.festivaldellafotografiaetica.it/organizzazione/",
    "https://www.festivaldellafotografiaetica.it/world-report-award-it/",
    "https://www.festivaldellafotografiaetica.it/sponsor/", # Per public_funding e international
    "https://www.festivaldellafotografiaetica.it/biglietteria/" # Per ticket_policy
]

def scrape_silos(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        # Pulizia del rumore
        for s in soup(['script', 'style', 'nav', 'footer', 'header']): s.extract()
        return soup.get_text(separator=' ', strip=True)
    except:
        return ""

print("Ampliamento del Corpus informativo in corso...")
corpus_totale = " ".join([scrape_silos(u) for u in urls])

# 3. DEFINIZIONE FRAMEWORK CON OPZIONI VINCOLATE
# Le domande ora guidano esplicitamente l'IA verso le opzioni ammesse
framework_attributi = {
    "festival_name": "What is the full name of the festival?",
    "city": "In which city does the festival take place?",
    "region": "In which Italian region is the city located?",
    "start_year": "In what year was the first edition held? (Answer with the year only, e.g. 2010)",
    "organizer_type": "What is the legal nature of the organizer? (Choose one: Non-profit, Public, Private)",
    "international": "Is there an international orientation or international partners? (Answer only: Yes or No)",
    "public_funding": "Is there presence of public funding or institutional sponsors? (Answer only: Yes or No)",
    "ticket_policy": "What is the entry type for the exhibitions? (Choose one: Free, Paid, Mixed)"
}

# 4. ESEMPI FEW-SHOT PER IL MODELLO
# Forniamo esempi concreti per istruire il modello sulla logica di risposta
esempi_guida = """
Esempio 1 - Testo: Il festival si tiene a Palermo, in Sicilia, ed è gestito da una ONLUS. Fondato nel 2005.
Question: city? Answer: Palermo.
Question: region? Answer: Sicilia.
Question: start_year? Answer: 2005.
Question: organizer_type? (Choose one: Non-profit, Public, Private) Answer: Non-profit.

Esempio 2 - Testo: Mostre gratuite aperte a tutti. Partner da USA e Giappone. Sponsor Ministero Cultura.
Question: international? (Answer only: Yes or No) Answer: Yes.
Question: public_funding? (Answer only: Yes or No) Answer: Yes.
Question: ticket_policy? (Choose one: Free, Paid, Mixed) Answer: Free.
"""

riga_dataset = {}

# 5. INFERENZA AUTOMATIZZATA FEW-SHOT
print("\n--- AVVIO ESTRAZIONE AUTOMATICA ---")

for chiave, domanda in framework_attributi.items():
    # Costruzione del prompt: Esempi + Contesto reale + Domanda specifica
    prompt = f"{esempi_guida}\n\nContext: {corpus_totale[:2500]}\nQuestion: {domanda}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        # Generazione della risposta (max 20 token per mantenere la sintesi)
        outputs = model.generate(**inputs, max_new_tokens=20)

    risposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    riga_dataset[chiave] = risposta
    print(f"Estratto [{chiave}]: {risposta}")

# 6. OUTPUT FINALE: DATASET NORMALIZZATO
df_final = pd.DataFrame([riga_dataset])
print("\n--- DATASET GENERATO (DALL'AGENTE) ---")
print(df_final.to_string())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Ampliamento del Corpus informativo in corso...


Token indices sequence length is longer than the specified maximum sequence length for this model (1126 > 512). Running this sequence through the model will result in indexing errors



--- AVVIO ESTRAZIONE AUTOMATICA ---
Estratto [festival_name]: Festival Della Fotografia Etica
Estratto [city]: Lodi
Estratto [region]: Italia
Estratto [start_year]: No
Estratto [organizer_type]: Non-profit
Estratto [international]: Yes
Estratto [public_funding]: Yes
Estratto [ticket_policy]: Paid

--- DATASET GENERATO (DALL'AGENTE) ---
                     festival_name  city  region start_year organizer_type international public_funding ticket_policy
0  Festival Della Fotografia Etica  Lodi  Italia         No     Non-profit           Yes            Yes          Paid


Criticità:
- errori di classificazione es. Italia al posto di Lombardia (SOLUZIONE: modifica del prompt per indicare di usare la propria conoscenza geografica, invece di cercare nel corpus testuale)
- limite di analisi a 512 token: approccio sliding window, dividendo il corpus in due parti

#Few-shot prova 2 + sliding window

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. CONFIGURAZIONE MODELLO
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

# 2. HARVESTING
urls = [
    "https://www.festivaldellafotografiaetica.it/",
    "https://www.festivaldellafotografiaetica.it/organizzazione/",
    "https://www.festivaldellafotografiaetica.it/world-report-award-it/",
    "https://www.festivaldellafotografiaetica.it/sponsor/",
    "https://www.festivaldellafotografiaetica.it/biglietteria/"
]

def scrape_silos(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        for s in soup(['script', 'style', 'nav', 'footer', 'header']): s.extract()
        return soup.get_text(separator=' ', strip=True)
    except: return ""

print("Harvesting dei silos informativi...")
corpus_totale = " ".join([scrape_silos(u) for u in urls])

# --- LOGICA DI SLIDING WINDOW ---
# Dividiamo il testo in due blocchi di circa 1800 caratteri (~500 token l'uno)
chunk_1 = corpus_totale[:1800]
chunk_2 = corpus_totale[1800:3600]
chunks = [chunk_1, chunk_2]

# 3. FRAMEWORK E ESEMPI
framework_attributi = {
    "festival_name": "What is the full name of the festival?",
    "city": "In which city does the festival take place?",
    "region": "Based on your general knowledge, in which Italian region is the city of Lodi located? (Answer with the region name only)",
    "start_year": "In what year was the first edition held? (Year only)",
    "organizer_type": "Legal nature? (Non-profit, Public, Private)",
    "international": "International orientation? (Yes/No)",
    "public_funding": "Presence of public funding? (Yes/No)",
    "ticket_policy": "Entry type? (Free, Paid, Mixed)"
}

esempi_guida = """
Esempio: Testo: Il festival si tiene a Lodi, Lombardia. Fondato nel 2010. Ingresso a pagamento.
Question: city? Answer: Lodi.
Question: start_year? Answer: 2010.
Question: ticket_policy? Answer: Paid.
"""

riga_dataset = {}

# 5. INFERENZA CON SLIDING WINDOW (Analisi per blocchi)
print("\n--- AVVIO ESTRAZIONE CON SLIDING WINDOW (2 CHUNKS) ---")

for chiave, domanda in framework_attributi.items():
    risposte_chunks = []

    for i, chunk in enumerate(chunks):
        prompt = f"{esempi_guida}\n\nContext: {chunk}\nQuestion: {domanda}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=20)

        res = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        risposte_chunks.append(res)

    # LOGICA DI AGGREGAZIONE (Max-Pooling concettuale)
    # Se è una domanda Yes/No e uno dei due chunk dice Yes, prendiamo Yes.
    # Altrimenti prendiamo la risposta più lunga o la prima non nulla.
    if "Yes" in risposte_chunks:
        valore_finale = "Yes"
    else:
        # Filtriamo risposte inutili come "No" o "Italia" se l'altro chunk ha dato qualcosa di meglio
        valid_responses = [r for r in risposte_chunks if r.lower() not in ["no", "italia", "unknown", "none"]]
        valore_finale = valid_responses[0] if valid_responses else risposte_chunks[0]

    riga_dataset[chiave] = valore_finale
    print(f"Analisi {chiave}: Chunk1='{risposte_chunks[0]}' | Chunk2='{risposte_chunks[1]}' -> Scelto: '{valore_finale}'")

# 6. OUTPUT
df_final = pd.DataFrame([riga_dataset])
print("\n--- DATASET GENERATO CON SLIDING WINDOW ---")
print(df_final.to_string())

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Harvesting dei silos informativi...


Token indices sequence length is longer than the specified maximum sequence length for this model (733 > 512). Running this sequence through the model will result in indexing errors



--- AVVIO ESTRAZIONE CON SLIDING WINDOW (2 CHUNKS) ---
Analisi festival_name: Chunk1='Festival Della Fotografia Etica' | Chunk2='Festival della Fotografia Etica' -> Scelto: 'Festival Della Fotografia Etica'
Analisi city: Chunk1='Lodi' | Chunk2='Lodi' -> Scelto: 'Lodi'
Analisi region: Chunk1='Lombardia' | Chunk2='Lombardia' -> Scelto: 'Lombardia'
Analisi start_year: Chunk1='2010' | Chunk2='2022' -> Scelto: '2010'
Analisi organizer_type: Chunk1='Non-profit' | Chunk2='Non-profit' -> Scelto: 'Non-profit'
Analisi international: Chunk1='No' | Chunk2='Yes' -> Scelto: 'Yes'
Analisi public_funding: Chunk1='No' | Chunk2='Yes' -> Scelto: 'Yes'
Analisi ticket_policy: Chunk1='Paid' | Chunk2='Paid' -> Scelto: 'Paid'

--- DATASET GENERATO CON SLIDING WINDOW ---
                     festival_name  city     region start_year organizer_type international public_funding ticket_policy
0  Festival Della Fotografia Etica  Lodi  Lombardia       2010     Non-profit           Yes            Yes          Paid


In [ ]:
print(corpus_totale)

Festival Della Fotografia Etica | XVI Edizione 2025 World Report Award | Documenting Humanity – Vincitori 2026 XVII Edizione: 26 settembre – 25 ottobre 2026 Festival della Fotografia Etica – Lodi – dal 2010 la fotografia necessaria World Report Award 2026 – Vincitori ONG Selezionate 2026 WRA 2026 – Shortlist Partnership con The Alexia Iscriviti alla Newsletter Bando Circuito OFF 2026 Chi siamo Edizione XVI 2025 Travelling Festival Dicono di noi Diventa Partner Press Area Progetti per le scuole Sostienici Scatti di Etica Scatti di Etica Podcast LETTURE PORTFOLIO Unisciti a noi, diventa volontario Photo Fun Mostre 2024 Pagina non trovata - Festival della Fotografia Etica Page not Found Go to Home page World Report Award - Festival della Fotografia Etica World Report Award Il World Report Award|Documenting Humanity si pone l’obiettivo di condividere una nuova forma di impegno sociale attraverso la fotografia. È aperto a tutti i fotografi professionisti italiani e non . Il soggetto è l’uom

Output corrisponde al Gold standard

# Estrazione testo da immagine OCR

In [ ]:
# 1. Installazione EasyOCR
!pip install easyocr
import os
import easyocr
import cv2
from google.colab.patches import cv2_imshow

# 2. Inizializzazione (Scarica i modelli per Italiano e Inglese)
reader = easyocr.Reader(['it', 'en'])

# 3. Estrazione Testo
path_img = "/locandina.jpg"

if os.path.exists(path_img):
    print("Estrazione OCR professionale in corso...")
    results = reader.readtext(path_img, detail=0) # detail=0 restituisce solo il testo

    full_text = " ".join(results)

    print("\n" + "═"*60)
    print("   VERO TESTO ESTRATTO (EasyOCR)")
    print("─"*60)
    print(full_text)
    print("═"*60)
else:
    print("File non trovato.")

Estrazione OCR professionale in corso...

════════════════════════════════════════════════════════════
   VERO TESTO ESTRATTO (EasyOCR)
────────────────────────────────────────────────────────────
FESTIVAL FOTOGRAFIA 27/26 ETICA SET OTT LODI 2025 XVI EDIZIONE WorlD Press PhoTo EPSON EERDOLARIO onorcom FUJIFILM CARTENI
════════════════════════════════════════════════════════════
